# Acervo que Fala — Notebook 01: o primeiro objeto de ponta a ponta

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

Este é o primeiro notebook do projeto. O objetivo dele é simples e único: fazer **um** objeto real do acervo do Museu do Índio atravessar o caminho completo — buscar a foto e os dados na API pública do museu, carregar um modelo de visão open source, e pedir que ele **observe a fotografia** seguindo as regras do projeto.

### Nota de metodologia

Sou designer, não desenvolvedor. Este projeto é construído com LLMs como suporte de desenvolvimento (a prática conhecida como *vibe coding*): o código foi escrito com auxílio do Claude, sob minha direção. Por isso, cada célula de código vem precedida de uma explicação em linguagem simples do que ela faz e por quê — executar sem entender não é o objetivo deste trabalho, e este notebook é também o registro de que cada etapa do processo foi compreendida.

### Como rodar

1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** (a placa gráfica gratuita do Colab — o modelo de visão precisa dela).
2. Menu **Ambiente de execução → Executar tudo**.
3. Tempo total: **~10 minutos** na primeira execução (a maior parte é o download do modelo).

O que sai no final: a **observação visual** do primeiro objeto — a matéria-prima que, nos próximos notebooks, vira o alt-text e a descrição acessível — salva automaticamente na pasta do projeto no Google Drive.

## Etapa 1 — Instalar as ferramentas (~2 min)

A célula abaixo instala as bibliotecas que o notebook usa:

- **transformers** — a biblioteca da Hugging Face que baixa e executa modelos abertos (é o "leitor" universal de modelos open source);
- **accelerate** — distribui o modelo entre a memória da GPU e do computador;
- **bitsandbytes** — permite a **quantização em 4-bit**: comprime os números internos do modelo de 16 para 4 bits, fazendo um modelo de 8 bilhões de parâmetros (~17 GB) caber nos 16 GB da GPU gratuita, com perda de qualidade pequena. Sem isso, precisaríamos pagar por uma GPU maior.

Vai aparecer bastante texto de instalação — é normal. Sucesso = a célula termina sem mensagem vermelha de erro.

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes pillow requests
print("ferramentas instaladas \u2713")

## Etapa 2 — Buscar um objeto real do acervo

O acervo digital do Museu do Índio roda na plataforma **Tainacan** (software público brasileiro) e tem uma **API pública** — um endereço na internet que devolve os dados de cada objeto em formato estruturado, sem precisar de senha.

A célula abaixo busca o objeto **9196 — um pote de cerâmica Karajá de 1977** (o primeiro objeto do smoke test do projeto): baixa a fotografia e os metadados curatoriais (povo, matéria-prima, técnica, dimensões...). Se a foto aparecer logo abaixo, a conexão com o museu funcionou.

In [ ]:
import io, re, requests
from PIL import Image
from IPython.display import display

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
ITEM_ID = 9196  # Pote Karajá — primeiro objeto do smoke test

item = requests.get(f"{BASE}/items/{ITEM_ID}", timeout=60).json()
url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))

meta_bruto = requests.get(f"{BASE}/item/{ITEM_ID}/metadata", timeout=60).json()
metadados = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}

display(foto)
for campo in ["Nome do item", "Povo", "Categoria", "Mat\u00e9ria-prima", "T\u00e9cnica de confec\u00e7\u00e3o", "Dimens\u00f5es", "Descri\u00e7\u00e3o"]:
    print(f"{campo}: {metadados.get(campo, '(vazio)')}")

## Etapa 3 — Carregar o modelo de visão (~5 min na primeira vez)

O modelo escolhido é o **Qwen3-VL-8B-Instruct** — um modelo aberto (licença Apache 2.0, uso livre) que "enxerga" imagens e escreve texto sobre elas. Foi escolhido por ser, na data do projeto, o melhor da categoria que roda em GPU gratuita; a comparação com alternativas está documentada no repositório.

A célula baixa ~6 GB (só na primeira vez — fica em cache) e carrega o modelo **quantizado em 4-bit** na GPU. Sucesso = a última linha imprime `modelo carregado ✓`.

> **Se der erro vermelho aqui**, a causa mais comum é a GPU não estar ativada (Etapa "Como rodar", passo 1). Se persistir, copie a última linha do erro e me envie no chat do Claude.

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"

quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = AutoModelForImageTextToText.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)
print("modelo carregado \u2713")

## Etapa 4 — A observação visual

Aqui está a regra mais importante do projeto. O modelo **não** recebe os metadados do museu nesta etapa — só a fotografia. E o prompt (a instrução) manda descrever **apenas o que está visível**, com uma cláusula de segurança: *"se algo estiver ilegível ou incerto, diga isso em vez de estimar"*.

Por quê? Modelos de visão têm um defeito conhecido chamado **alucinação visual**: descrevem o que *tipicamente* estaria numa cena, não necessariamente o que está. Essa instrução é o plano de contingência contra isso — e separar a observação (esta etapa) da redação final (próximos notebooks) permite verificar cada parte do processo separadamente.

In [ ]:
PROMPT_OBSERVACAO = (
    "Descreva APENAS o que est\u00e1 vis\u00edvel nesta fotografia de um objeto de museu: "
    "formas, cores, materiais aparentes, texturas, enquadramento (o objeto aparece "
    "inteiro ou s\u00f3 um detalhe?), fundo, e qualquer artefato de est\u00fadio (etiqueta, "
    "numera\u00e7\u00e3o, cartela de cores, r\u00e9gua, suporte). "
    "N\u00c3O invente o que n\u00e3o d\u00e1 para ver. Se algo estiver ileg\u00edvel ou incerto, "
    "diga isso em vez de estimar. Responda em portugu\u00eas."
)

conversa = [{
    "role": "user",
    "content": [
        {"type": "image", "image": foto},
        {"type": "text", "text": PROMPT_OBSERVACAO},
    ],
}]

entradas = processador.apply_chat_template(
    conversa, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(modelo.device)

with torch.no_grad():
    saida = modelo.generate(**entradas, max_new_tokens=400)

observacao = processador.decode(
    saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True
)
print(observacao)

## Etapa 5 — Conferir com olhos humanos

A máquina observou; agora é a parte humana do processo. Compare o texto acima com a fotografia e responda:

1. O modelo descreveu **só o que se vê** (pote, gargalo, grafismos em vermelho e preto sobre argila crua, fundo branco)?
2. Ele **inventou** alguma coisa que não está na foto?
3. Ele avisou o **enquadramento** (o pote aparece inclinado, com a boca visível)?

Repare que o modelo não sabe que isto é um pote Karajá de 1977 — e não deveria dizer isso: essa informação vem dos metadados do museu, que entram só na etapa de **redação** (próximo notebook), sempre com fonte citada.

## Etapa 6 — Salvar o resultado no Drive

Para manter o projeto organizado, cada execução salva seu resultado na pasta do projeto no Google Drive (`00_IA / GenAI & LLMs - PUC / Projeto_LLM / resultados`), com nome padronizado.

A célula abaixo pede **autorização para conectar ao SEU Google Drive** — vai abrir uma janela; clique em *Conectar ao Google Drive* e escolha sua conta. É o seu próprio Drive; nada sai da sua conta.

In [ ]:
import json, os
from google.colab import drive

drive.mount("/content/drive")

PASTA = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM/resultados"
os.makedirs(PASTA, exist_ok=True)

resultado = {
    "notebook": "01_primeiro_item",
    "item_id": ITEM_ID,
    "modelo": MODELO,
    "prompt": PROMPT_OBSERVACAO,
    "observacao": observacao,
}
destino = os.path.join(PASTA, "01_observacao_item_9196.json")
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive \u2713  {destino}")

---

## Fim — o que fazer agora

Copie a **observação** impressa na Etapa 4 e cole no chat do Claude (o arquivo também ficou salvo no Drive). A análise dela define os ajustes de prompt antes de rodar os 5 objetos do smoke test no próximo notebook.

**O que este notebook provou:** o ambiente gratuito executa o modelo de visão do projeto, a API do museu responde, o prompt anti-alucinação produz uma observação verificável, e os resultados ficam organizados no Drive. **O que ainda não provou:** qualidade em escala (40 casos), os dois níveis de descrição, e a comparação com a baseline — assunto dos próximos notebooks.